In [1]:
"""
ao_comp.py -- Complexity estimation for Dixon-based attacks on AO primitives.

Pure Python (no Sage).  Exact integer arithmetic throughout; only the final
log2 is floating point.

Corrections w.r.t. the original AO_Comp.sage
--------------------------------------------
(C1) Dixon cost model now takes the number of RETAINED parameters k explicitly
     and implements both Step-4 models of the paper:
         HNF  : M^w * E * (M*E+1)^(k-1)          [Kronecker + Hermite normal form]
         eval : (D_out+1)^k * (M^w + k*D_out)    [evaluation / interpolation]
     and reports min(HNF, eval).  The old code used (sum(d)+1)^(nvars-m+1),
     which drops the factor M inside the Kronecker base.

(C2) "Ladder" (Method 2) systems are costed STAGE BY STAGE (every forward and
     backward elimination + the final merge), and the reported cost is the max
     over stages, not just the merge.

(C3) Vision: the meet-in-the-middle cut is optimised over all admissible block
     positions instead of hard-coding d_max = 2^(4r).  This reproduces
     Table 9 of the paper: d_max = 2^(4r) for even r, 2^(4r-2) for odd r.

(C4) Ladder steps use the block-degree refinement: in {R_{i-1}} u F_i the
     polynomial R_{i-1} does NOT involve the retained block X_i, so every entry
     of the Dixon matrix has X_i-degree at most sum_{f in F_i} deg_{X_i}(f).
     The old model used deg(R_{i-1}) + sum(deg f), which is far too large.

(C5) XHash Method 3 now costs all three phases P1 (reduction), P2 (successive
     elimination) and P3 (final mixed-degree Dixon), following (27)-(29) of the
     revised paper, instead of P3 alone.
"""

from math import comb, log2, lgamma
from fractions import Fraction

LOG2E = 1.4426950408889634

# ---------------------------------------------------------------------------
# big-integer safe log2
# ---------------------------------------------------------------------------

def flog2(x):
    """log2 of a positive (possibly huge) integer or float."""
    if isinstance(x, Fraction):
        return flog2(x.numerator) - flog2(x.denominator)
    if isinstance(x, float):
        return log2(x)
    x = int(x)
    if x <= 0:
        return float('-inf')
    b = x.bit_length()
    if b <= 512:
        return log2(x)
    # mantissa with 64 bits of precision
    shift = b - 64
    return (b - 1) + log2((x >> shift) / float(1 << 63))


# ---------------------------------------------------------------------------
# Dixon matrix support-size bounds  (Section 3.1 / Corollary 1)
# ---------------------------------------------------------------------------

def fuss_catalan(n, d):
    """C_n^(d) = binom(nd, n) / (n(d-1)+1).  Exact integer."""
    if n <= 0 or d <= 0:
        return 1
    num = comb(n * d, n)
    den = n * (d - 1) + 1
    assert num % den == 0
    return num // den


def _int_det(M):
    """Exact integer determinant by Bareiss (fraction-free) elimination."""
    n = len(M)
    if n == 0:
        return 1
    A = [row[:] for row in M]
    sign = 1
    prev = 1
    for k in range(n - 1):
        if A[k][k] == 0:
            for i in range(k + 1, n):
                if A[i][k] != 0:
                    A[k], A[i] = A[i], A[k]
                    sign = -sign
                    break
            else:
                return 0
        for i in range(k + 1, n):
            for j in range(k + 1, n):
                A[i][j] = (A[i][j] * A[k][k] - A[i][k] * A[k][j]) // prev
        prev = A[k][k]
    return sign * A[n - 1][n - 1]


def dixon_size_determinant(degs):
    """Corollary 1(3): ordered mixed-total-degree (Hessenberg) determinant bound.

    degs is the multiset of total degrees of the n polynomials; n-1 variables
    are eliminated.  Returns the support-size bound D(F).
    """
    d = sorted(degs)                      # d_1 <= ... <= d_n
    n = len(d)
    if n <= 1:
        return 1
    m = n - 1
    a = []
    s = 0
    for i in range(1, m + 1):
        s += d[n - i] - 1                 # d_{n+1-i} - 1
        a.append(s)
    M = [[comb(a[i] + 1, j - i + 1) if 0 <= j - i + 1 <= a[i] + 1 else 0
          for j in range(m)] for i in range(m)]
    return _int_det(M)


def dixon_size_unrestricted(degs):
    """Corollary 1(1): binom(sum_{i>=2} d_i, n-1)."""
    d = sorted(degs)
    n = len(d)
    if n <= 1:
        return 1
    return comb(sum(d[1:]), n - 1)


def dixon_support_size(degs):
    """Best available support bound: Fuss-Catalan if uniform, else Cor 1(3)."""
    d = sorted(degs)
    if len(d) <= 1:
        return 1
    if d[0] == d[-1]:
        return fuss_catalan(len(d), d[0])
    return min(dixon_size_determinant(d), dixon_size_unrestricted(d))


# ---------------------------------------------------------------------------
# Step-4 cost models  (Section 3.2)
# ---------------------------------------------------------------------------

def step4_hnf(M, k, E, omega):
    """Kronecker + HNF:  M^w * E * (M*E+1)^(k-1)   (log2)."""
    return omega * flog2(M) + flog2(E) + (k - 1) * flog2(M * E + 1)


def step4_eval(M, k, D_out, omega):
    """Evaluation / interpolation:  (D_out+1)^k * (M^w + k*D_out)   (log2)."""
    grid = k * flog2(D_out + 1)
    probe = max(omega * flog2(M), flog2(k * D_out + 1))
    return grid + probe


def dixon_elimination_cost(degs, k, omega,
                           retained_degs=None, D_out=None, model="min"):
    """Cost of ONE Dixon elimination.

    degs           : total degrees of the n polynomials (n-1 vars eliminated)
    k              : number of retained parameters
    retained_degs  : per-polynomial degree in the RETAINED block.  If None,
                     falls back to `degs` (the conservative choice used in the
                     original code).  Supplying the true block degrees is the
                     refinement (C4) above.
    D_out          : total degree of the output (Bezout by default)
    Returns (log2 cost, M, E, D_out).
    """
    M = dixon_support_size(degs)
    rd = degs if retained_degs is None else retained_degs
    E = max(1, sum(rd))
    if D_out is None:
        D_out = 1
        for x in degs:
            D_out *= x
    c_hnf = step4_hnf(M, k, E, omega)
    c_evl = step4_eval(M, k, D_out, omega)
    if model == "hnf":
        c = c_hnf
    elif model == "eval":
        c = c_evl
    else:
        c = min(c_hnf, c_evl)
    return c, M, E, D_out


# ---------------------------------------------------------------------------
# Groebner-basis reference costs
# ---------------------------------------------------------------------------

def gb_f4(n_eqs, dreg, omega):
    return omega * flog2(comb(n_eqs + dreg, n_eqs))


def gb_fglm(n_vars, dI, omega):
    return flog2(n_vars) + omega * flog2(dI)


# ===========================================================================
#  VISION  (Method 2: ladder / meet-in-the-middle)
# ===========================================================================
#
# For r rounds and s = 2 branches the CICO system has 4r-1 variables organised
# as one singleton block plus 2r-1 pairs, and 4r-1 equations organised as
# 2r-1 "rungs" plus one output constraint of degree 4:
#
#   rung 1 : (2,2)   rung 2 : (8,8)   rung 3 : (2,2)   rung 4 : (8,8)  ...
#
# (verified against Vision/vision-{1,2,3,4}r.sage).  Total Bezout = 2^(8r-4).

def vision_rungs(r):
    """List of (deg, deg) rung degree pairs, plus the trailing constraint deg."""
    rungs = []
    for i in range(1, 2 * r):
        rungs.append((2, 2) if i % 2 == 1 else (8, 8))
    return rungs, 4


def vision_cut_profile(r):
    """Forward/backward accumulated degrees for every admissible cut."""
    rungs, tail = vision_rungs(r)
    total = tail
    for a, b in rungs:
        total *= a * b
    prof = []
    fwd = 1
    for c, (a, b) in enumerate(rungs, start=1):
        fwd *= a * b
        bwd = total // fwd
        prof.append((c, fwd, bwd, max(fwd, bwd)))
    return prof, total


def vision_cost(r, omega=2.0, cut=None, block_degree=True, verbose=False):
    """Full stage-wise cost of the Vision MITM attack."""
    rungs, tail = vision_rungs(r)
    prof, total = vision_cut_profile(r)
    if cut is None:
        cut = min(prof, key=lambda t: t[3])[0]
    stages = []

    # ---- forward chain: rungs 1..cut ----------------------------------------
    D = 1
    for i in range(1, cut + 1):
        a, b = rungs[i - 1]
        if i == 1:
            # first rung eliminates the singleton block: two polynomials,
            # one variable -> Bezout/Cayley, matrix size max(a,b)
            M, E, k = max(a, b), a + b, 2
            c = step4_hnf(M, k, E, omega)
        else:
            degs = [D, a, b]
            rdeg = [0, a, b] if block_degree else degs
            c, M, E, _ = dixon_elimination_cost(degs, 2, omega,
                                                retained_degs=rdeg,
                                                D_out=D * a * b, model="hnf")
        D *= a * b
        stages.append((f"fwd rung {i} (deg {a},{b})", c, M, D))
    D_fwd = D

    # ---- backward chain: tail + rungs cut+1..2r-1 (from the output side) ----
    D = tail
    stages.append((f"bwd constraint (deg {tail})", 0.0, 1, D))
    for i in range(len(rungs), cut, -1):
        a, b = rungs[i - 1]
        degs = [D, a, b]
        rdeg = [0, a, b] if block_degree else degs
        c, M, E, _ = dixon_elimination_cost(degs, 2, omega,
                                            retained_degs=rdeg,
                                            D_out=D * a * b, model="hnf")
        D *= a * b
        stages.append((f"bwd rung {i} (deg {a},{b})", c, M, D))
    D_bwd = D

    # ---- final merge: two bivariate relations, eliminate one variable -------
    d_max = max(D_fwd, D_bwd)
    M_m = d_max
    E_m = D_fwd + D_bwd
    c_merge = step4_hnf(M_m, 1, E_m, omega)
    stages.append(("MERGE", c_merge, M_m, d_max))

    best = max(s[1] for s in stages)
    if verbose:
        print(f"  Vision r={r}, cut after rung {cut}, "
              f"D_fwd=2^{flog2(D_fwd):.0f}, D_bwd=2^{flog2(D_bwd):.0f}, "
              f"d_max=2^{flog2(d_max):.0f}")
        for name, c, M, D in stages:
            print(f"    {name:28s} M=2^{flog2(M):7.2f}  "
                  f"D=2^{flog2(D):8.2f}  cost=2^{c:7.2f}")
    return {"cut": cut, "d_max": d_max, "D_fwd": D_fwd, "D_bwd": D_bwd,
            "merge": c_merge, "max_stage": best, "stages": stages}


def vision_gb_theoretical(r, omega):
    """Groebner reference used in the original script."""
    n = 4 * r - 1
    n8, n2, n4 = 2 * (r - 1), 2 * r, 1
    dreg = n8 * 7 + n4 * 3 + n2 * 1 + 1
    return gb_f4(n, dreg, omega)


# ===========================================================================
#  XHASH12  (Method 1 direct, Method 3 hybrid)
# ===========================================================================

def xhash_method1(n_levels, k, alpha, omega):
    """Direct Dixon on the unreduced system of n+k equations of degree alpha,
    eliminating n+k-1 variables (one retained parameter)."""
    m = n_levels + k
    M = fuss_catalan(m, alpha)
    E = m * alpha
    return step4_hnf(M, 1, E, omega), M


def xhash_method3(n_levels, k, alpha, omega, D0=1, verbose=False):
    """Hybrid reduction + Dixon, phases P1/P2/P3 of the revised paper.

    P1  T_red   = k * B_k(D0) * (2a-1)^n
    P2  T_elim  = k * sum_{j=1..n} B_k(a^(n-j+1) * D0) * (2a-1)^(j+2)
    P3  T_final = k * D * D(R)^omega,  D = a^n * D0
    with B_k(D) = binom(D+k, k).
    """
    a = alpha
    n = n_levels
    Bk = lambda D: comb(D + k, k)

    t_red = flog2(k) + flog2(Bk(D0)) + n * flog2(2 * a - 1)

    best_j, t_elim = None, float('-inf')
    for j in range(1, n + 1):
        Dj = a ** (n - j + 1) * D0
        c = flog2(Bk(Dj)) + (j + 2) * flog2(2 * a - 1)
        if c > t_elim:
            t_elim, best_j = c, j
    t_elim += flog2(k * n)          # sum over stages and constraints

    D = a ** n * D0
    MR = dixon_support_size([D] * k) if k >= 2 else 1
    t_fin = (flog2(k) + flog2(D) + omega * flog2(MR)) if k >= 2 else \
            flog2(D)                # k=1: the relation is already univariate
    if verbose:
        print(f"    n={n} k={k} a={a}:  P1=2^{t_red:.1f}  "
              f"P2=2^{t_elim:.1f} (worst stage j={best_j})  "
              f"P3=2^{t_fin:.1f}  (M_R=2^{flog2(MR):.1f})")
    return {"P1": t_red, "P2": t_elim, "P3": t_fin,
            "total": max(t_red, t_elim, t_fin), "MR": MR, "D": D}


def xhash_cico1_twopoly(n_levels, alpha, omega):
    """Eq. (26): the two-polynomial FreeLunch route, CICO-1 only."""
    dI = alpha ** n_levels
    return flog2(dI) + 2 * flog2(alpha) + (n_levels + 2) * (flog2(2 * alpha - 1)
                                                            - flog2(alpha))


def xhash_gb(n_levels, k, alpha, omega):
    m = n_levels + k
    dreg = m * (alpha - 1) + 1
    return gb_f4(m, dreg, omega)


# ===========================================================================
#  Multihomogeneous Bezout  (block-structured / ladder systems)
# ===========================================================================
#
# For a system whose equations have prescribed degrees in each variable BLOCK,
# the number of isolated solutions is bounded by the coefficient of
# prod_j z_j^{k_j} in prod_i (sum_j d_ij z_j), where k_j = |B_j|.
# For the Vision ladder this is much smaller than the plain Bezout product.

def mh_bezout(eqs, block_sizes):
    """eqs: list of dicts {block index -> degree in that block}."""
    nb = len(block_sizes)
    poly = {tuple([0] * nb): 1}
    for e in eqs:
        out = {}
        for mono, c in poly.items():
            for j, dg in e.items():
                if dg == 0 or mono[j] + 1 > block_sizes[j]:
                    continue
                m2 = list(mono)
                m2[j] += 1
                m2 = tuple(m2)
                out[m2] = out.get(m2, 0) + c * dg
        poly = out
        if not poly:
            return 0
    return poly.get(tuple(block_sizes), 0)


def vision_ladder_eqs(r):
    """Rung equations of the Vision CICO system as multidegrees.

    Blocks: B_0 = {x_0} (size 1), B_1..B_{2r-1} (size 2).
    Rung i joins B_{i-1} and B_i with two equations of bidegree (d_i/2, d_i/2);
    d_i = 2 for odd i, 8 for even i.  One trailing constraint of degree 4 on
    B_{2r-1}.  Verified against Vision/vision-{1,2,3,4}r.sage.
    """
    rg = [2 if i % 2 == 1 else 8 for i in range(1, 2 * r)]
    return rg


def vision_deg_mh(r, cut, side, per_variable=False):
    """Multihomogeneous degree bound of the forward / backward eliminant.

    side = 'fwd' : eliminate B_0..B_{cut-1}, retain B_cut
    side = 'bwd' : eliminate B_{2r-1}..B_{cut+1}, retain B_cut
    per_variable : bound deg in ONE variable of B_cut (the other being fixed)
    """
    rg = vision_ladder_eqs(r)
    if side == 'fwd':
        last = 1 if per_variable else 2
        ksz = [1] + [2] * (cut - 1) + [last]
        eqs = []
        for i in range(1, cut + 1):
            d = rg[i - 1]
            eqs += [{i - 1: d // 2, i: d // 2}] * 2
        if not per_variable:
            eqs.append({cut: 1})           # generic line in B_cut
        return mh_bezout(eqs, ksz)
    else:
        nb = len(rg) - cut + 1
        first = 1 if per_variable else 2
        ksz = [first] + [2] * (nb - 1)
        eqs = []
        for i in range(cut + 1, len(rg) + 1):
            d = rg[i - 1]
            eqs += [{i - 1 - cut: d // 2, i - cut: d // 2}] * 2
        eqs.append({nb - 1: 4})            # output constraint
        if not per_variable:
            eqs.append({0: 1})
        return mh_bezout(eqs, ksz)


def vision_bezout_split(r, cut):
    """Plain Bezout accumulated degrees (the model of the original script)."""
    rg = vision_ladder_eqs(r)
    total = 4
    for d in rg:
        total *= d * d
    fwd = 1
    for i in range(cut):
        fwd *= rg[i] * rg[i]
    return fwd, total // fwd


def vision_merge_cost(r, omega=2.0, degree_model="mh"):
    """Cost of the final merge under the chosen degree model, optimising the cut."""
    rg = vision_ladder_eqs(r)
    best = None
    for cut in range(1, len(rg) + 1):
        if degree_model == "mh":
            Df, Db = vision_deg_mh(r, cut, 'fwd'), vision_deg_mh(r, cut, 'bwd')
            uf, ub = (vision_deg_mh(r, cut, 'fwd', True),
                      vision_deg_mh(r, cut, 'bwd', True))
        else:
            Df, Db = vision_bezout_split(r, cut)
            uf, ub = Df, Db
        if min(Df, Db) == 0:
            continue
        key = max(Df, Db)
        if best is None or key < best[0]:
            best = (key, cut, Df, Db, uf, ub)
    _, cut, Df, Db, uf, ub = best
    M = max(uf, ub)                 # Bezout-Cayley matrix size in the eliminated var
    E = uf + ub                     # column degree in the retained var
    cost = omega * flog2(M) + flog2(E)
    return {"cut": cut, "D_fwd": Df, "D_bwd": Db, "d_max": max(Df, Db),
            "M": M, "E": E, "cost": cost}


# ===========================================================================
#  Report
# ===========================================================================

def report():
    print("=" * 90)
    print("VISION (s=2): merge cost under three degree models")
    print("=" * 90)
    header = "MH w=2.81"
    print(f"{'r':>3} | {'old script':>10} | {'parity fix':>10} | "
          f"{'MH w=2':>8} | {header:>9} | {'GB [LMM24]':>10}")
    liu = {2: 31, 3: 42, 4: 53, 5: 64, 6: 74, 7: 85, 8: 95}
    for r in range(2, 15):
        prof, _ = vision_cut_profile(r)
        dm = min(prof, key=lambda t: t[3])[3]
        print(f"{r:>3} | {12*r+1:>10} | {3*flog2(dm)+1:>10.1f} | "
              f"{vision_merge_cost(r, 2.0, 'mh')['cost']:>8.1f} | "
              f"{vision_merge_cost(r, 2.81, 'mh')['cost']:>9.1f} | "
              f"{liu.get(r, '-'):>10}")
    print("\n  d_max:  Bezout -> 2^(4r) / 2^(4r-2);   MH-Bezout -> 2^(3r) / 2^(3r-1)")

    print()
    print("=" * 90)
    print("VISION: stage-by-stage (r=4, omega=2, plain-Bezout degrees)")
    print("=" * 90)
    vision_cost(4, omega=2.0, verbose=True)

    print()
    print("=" * 90)
    print("XHASH12 toy instances (equation counts read from the .dr artifacts)")
    print("=" * 90)
    for name, degs, note in [
            ("CICO-1 (2 step)", [3] * 6, "6 eq deg 3"),
            ("CICO-1 (3 step)", [7] * 7, "7 eq deg 7"),
            ("CICO-2 (2 step)", [3] * 6, "6 eq deg 3")]:
        M = dixon_support_size(degs)
        print(f"  {name:18s} M={M:>9}  Method1 = 2^"
              f"{step4_hnf(M, 1, sum(degs), 2.81):5.2f}   [{note}]")
    D = 3 ** 5
    print(f"  Method 1+3 P3 (CICO-2, 2 step, D=alpha^(n+1)={D}): 2^"
          f"{flog2(2) + flog2(D) + 2.81*flog2(dixon_support_size([D, D])):.2f}")

    print()
    print("=" * 90)
    print("XHASH12 full parameters (alpha=7, n=12s): Method 3 phases vs Method 1")
    print("=" * 90)
    print(f"{'s':>2} {'k':>2} | {'P1':>7} {'P2':>8} {'P3':>9} {'M3':>9} | {'M1':>8} | {'best':>5}")
    for s in [1, 2, 3, 4]:
        for k in [1, 2, 3, 4]:
            r = xhash_method3(12 * s, k, 7, 2.81, D0=7)
            m1, _ = xhash_method1(12 * s, k, 7, 2.81)
            print(f"{s:>2} {k:>2} | {r['P1']:>7.1f} {r['P2']:>8.1f} {r['P3']:>9.1f} "
                  f"{r['total']:>9.1f} | {m1:>8.1f} | "
                  f"{'M3' if r['total'] < m1 else 'M1':>5}")
        print()

if __name__ == "__main__":
    report()


VISION (s=2): merge cost under three degree models
  r | old script | parity fix |   MH w=2 | MH w=2.81 | GB [LMM24]
  2 |         25 |       25.0 |     18.2 |      23.0 |         31
  3 |         37 |       31.0 |     24.6 |      31.1 |         42
  4 |         49 |       49.0 |     36.2 |      45.9 |         53
  5 |         61 |       55.0 |     42.6 |      53.9 |         64
  6 |         73 |       73.0 |     54.2 |      68.7 |         74
  7 |         85 |       79.0 |     60.6 |      76.8 |         85
  8 |         97 |       97.0 |     72.2 |      91.6 |         95
  9 |        109 |      103.0 |     78.6 |      99.6 |          -
 10 |        121 |      121.0 |     90.2 |     114.5 |          -
 11 |        133 |      127.0 |     96.6 |     122.5 |          -
 12 |        145 |      145.0 |    108.2 |     137.3 |          -
 13 |        157 |      151.0 |    114.6 |     145.4 |          -
 14 |        169 |      169.0 |    126.2 |     160.2 |          -

  d_max:  Bezout -> 2^(4

In [2]:
"""Vision ladder: three cost models for the chain rungs."""

def rung_dixon_generic(D_in, d, D_out, omega):
    """(A) Paper's current model: uses total degree, E = D_in + 2d, Kronecker base = M*E"""
    M = dixon_support_size([d, d, D_in]) if D_in > 1 else 1
    E = D_in + 2 * d
    return omega * flog2(M) + flog2(E) + flog2(M * E + 1)

def rung_dixon_refined(D_in, d, D_out, omega):
    """(B) Lever 1+2: eliminated block degree d/2, E = d, base min(M*E, D_out)"""
    de = d // 2
    M = dixon_support_size([de, de, D_in]) if D_in > 1 else 1
    E = 2 * de
    return omega * flog2(M) + flog2(E) + flog2(min(M * E, D_out) + 1)

def rung_substitution(D_in, d, D_out, omega):
    """(C) Bilinear substitution: output-size driven, ~ D_out^2 coefficient"""
    return 2 * flog2(max(D_out, 2))

RUNG = {"generic": rung_dixon_generic,
        "refined": rung_dixon_refined,
        "subst":   rung_substitution}

def vision_stages(r, omega=2.0, model="refined", degree="mh", cut=None):
    rg = vision_ladder_eqs(r)
    def Df(c): return (vision_deg_mh(r, c, 'fwd') if degree == "mh"
                       else vision_bezout_split(r, c)[0]) if c >= 1 else 1
    def Db(c): return (vision_deg_mh(r, c, 'bwd') if degree == "mh"
                       else vision_bezout_split(r, c)[1])
    if cut is None:
        cut = min(range(1, len(rg) + 1), key=lambda c: max(Df(c), Db(c)))
    f = RUNG[model]
    st = []
    for i in range(1, cut + 1):
        st.append((f"fwd{i}(d={rg[i-1]})",
                   0.0 if i == 1 else f(Df(i - 1), rg[i - 1], Df(i), omega)))
    for i in range(len(rg), cut, -1):
        st.append((f"bwd{i}(d={rg[i-1]})", f(Db(i), rg[i - 1], Db(i - 1), omega)))
    m = vision_merge_cost(r, omega, 'mh' if degree == "mh" else 'bezout')
    st.append(("MERGE", m['cost']))
    return st, cut

def vision_total(r, omega=2.0, model="refined", degree="mh"):
    st, _ = vision_stages(r, omega, model, degree)
    return max(c for _, c in st)

def vision_merge_only(r, omega=2.0, degree="mh"):
    return vision_merge_cost(r, omega, 'mh' if degree == "mh" else 'bezout')['cost']


def rung_rho(D_in, d):
    """Bound on the size of the maximal-rank submatrix of a ladder rung.

    Appendix K, rung rank bound (proved, not fitted): for an incoming relation of total
    degree D_in and two local equations of degree e in the eliminated block,

        rank M  <=  e(e+3)/2 * D_in + e(3e-1)/2,

    with no genericity and no assumption on char(K), so it is valid over
    GF(2^l) as well.  We take the minimum with the support dimension, which is
    always an upper bound for the rank.  For the first rung (D_in <= 1) there is
    no incoming relation: the elimination is a Bezout-Cayley resultant of the
    two local equations and the matrix size is E = 2e.
    """
    e = d // 2
    E = 2 * e
    if D_in <= 1:
        return max(1, E)
    M = dixon_support_size([e, e, D_in])
    return min(M, e * (e + 3) // 2 * D_in + e * (3 * e - 1) // 2)


def rung_rank_aware(D_in, d, D_out, omega):
    """(D) rank-aware Dixon: Step 4 works on the maximal-rank submatrix of size
       rho given by rung_rho, instead of the support dimension M.  Step 3 uses a
       randomized rank-revealing projection: O(M * rho^(omega-1))."""
    e = d // 2
    E = 2 * e
    r_ = rung_rho(D_in, d)
    M = dixon_support_size([e, e, D_in]) if D_in > 1 else 1
    t3 = flog2(M) + (omega - 1) * flog2(r_)
    t4 = omega * flog2(r_) + flog2(E) + flog2(min(r_ * E, D_out) + 1)
    return max(t3, t4)


RUNG["rank"] = rung_rank_aware


def rung_full(D_in, d, D_out, omega):
    """(E) Complete three-stage model used for Tables 10-11 and Figure 4:
         Step 1  Delta has O(e^4 D_in^3) terms (same lemma), far below
                 the Theta(M^2) coefficient positions of a dense Dixon polynomial
         Step 3  randomized rank-revealing projection: O(M * rho^(omega-1))
         Step 4  uses rho = rung_rho(D_in, d) instead of the support dimension M
    """
    e = d // 2
    E = 2 * e
    r_ = rung_rho(D_in, d)
    M = dixon_support_size([e, e, D_in]) if D_in > 1 else 1
    t1 = 3 * flog2(max(D_in, 2))
    t3 = flog2(M) + (omega - 1) * flog2(r_)
    t4 = omega * flog2(r_) + flog2(E) + flog2(min(r_ * E, D_out) + 1)
    return max(t1, t3, t4)


RUNG["full"] = rung_full

In [3]:
"""
ao_plots.py -- curves for the AO applications, pure Python.

    python3 ao_plots.py                 # all tables + all PNGs

Vision: every model variant is kept as a NAMED SERIES so you can pick a subset
for the camera-ready figure without editing any formula.  Choose with

    plot_vision(series=["ao_comp", "bez_rank", "gb_overdef"])

Series registry (see VISION_SERIES below):
    ao_comp      12r+1, i.e. the original AO_Comp.sage curve (d_max = 2^(4r))
    bez_generic  plain Bezout degrees, generic Dixon bound, max over stages
    bez_rank     plain Bezout degrees, rank-aware Step1/3/4, max over stages
    bez_merge    plain Bezout degrees, merge stage only
    mh_generic   multihomogeneous Bezout degrees, generic bound
    mh_rank      multihomogeneous Bezout degrees, rank-aware
    mh_merge     multihomogeneous Bezout degrees, merge only
    gb_theory    Groebner (theoretical, F4 bound)
    gb_overdef   Groebner (overdefined, measured, [LMM24])

MINIMAL CAMERA-READY SET:  ["ao_comp", "bez_rank", "gb_theory", "gb_overdef"]
    -- keeps the paper's plain-Bezout degrees, no multihomogeneous machinery,
       and only needs the short rank remark added to Section 5.2.
"""

from math import floor, sqrt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SERIF = {"font.family": "serif",
         "font.serif": ["Liberation Serif", "DejaVu Serif"],
         "mathtext.fontset": "dejavuserif"}

GB_OVERDEF = {2: 31, 3: 42, 4: 53, 5: 64, 6: 74, 7: 85, 8: 95, 10: 115,
              12: 136, 14: 156, 16: 176, 18: 197, 20: 217, 22: 237,
              24: 250, 26: 271}


def vision_gb_theory(r, omega):
    n = 4 * r - 1
    dreg = 2 * (r - 1) * 7 + 3 + 2 * r + 1
    return gb_f4(n, dreg, omega)


# name -> (callable(r, omega), label, style dict)
VISION_SERIES = {
    "ao_comp":     (lambda r, w: 12 * r + 1,
                    r"Dixon, AO\_Comp.sage ($d_{\max}=2^{4r}$)",
                    dict(color="gray", ls=":", marker="x", lw=2)),
    "bez_generic": (lambda r, w: vision_total(r, w, "generic", "bezout"),
                    "Dixon, Bezout deg., generic bound (max over stages)",
                    dict(color="darkorange", ls="-", marker="v", lw=2)),
    "bez_rank":    (lambda r, w: vision_total(r, w, "full", "bezout"),
                    "Dixon, Bezout deg., rank-aware (max over stages)",
                    dict(color="red", ls="-", marker="o", lw=3)),
    "bez_merge":   (lambda r, w: vision_merge_only(r, w, "bezout"),
                    "Dixon, Bezout deg., merge stage only",
                    dict(color="red", ls="--", marker="", lw=1.5, alpha=.6)),
    "mh_generic":  (lambda r, w: vision_total(r, w, "generic", "mh"),
                    "Dixon, MH-Bezout deg., generic bound",
                    dict(color="purple", ls="-", marker="d", lw=2)),
    "mh_rank":     (lambda r, w: vision_total(r, w, "full", "mh"),
                    "Dixon, MH-Bezout deg., rank-aware",
                    dict(color="magenta", ls="-", marker="s", lw=2)),
    "mh_merge":    (lambda r, w: vision_merge_only(r, w, "mh"),
                    "Dixon, MH-Bezout deg., merge stage only",
                    dict(color="magenta", ls="--", marker="", lw=1.5, alpha=.6)),
    "gb_theory":   (vision_gb_theory,
                    "Groebner (theoretical)",
                    dict(color="blue", ls="-", marker="^", lw=2, alpha=.6)),
}

CAMERA_READY = ["ao_comp", "bez_rank", "gb_theory", "gb_overdef"]


def vision_curves(max_r=27, omega=2.0, series=None):
    series = list(VISION_SERIES) if series is None else \
        [s for s in series if s in VISION_SERIES]
    rs = list(range(2, max_r))
    return rs, {s: [VISION_SERIES[s][0](r, omega) for r in rs] for s in series}


def plot_vision(fn="vision_complexity.png", max_r=27, omega=2.0,
                series=None, with_overdef=True, ymax=350):
    plt.rcParams.update(SERIF)
    rs, cur = vision_curves(max_r, omega, series)
    plt.figure(figsize=(11, 6.5))
    for name, ys in cur.items():
        _, lab, st = VISION_SERIES[name]
        plt.plot(rs, ys, markevery=2, label=lab, **st)
    if with_overdef and (series is None or "gb_overdef" in series):
        ks = sorted(GB_OVERDEF)
        plt.plot(ks, [GB_OVERDEF[k] for k in ks], "^-", color="green", lw=2,
                 label="Groebner (overdefined, [LMM24])")
    for y, lab in [(128, "128-bit"), (256, "256-bit")]:
        plt.axhline(y, color="gray", ls="--", lw=1, alpha=.7)
        plt.text(max_r - 4, y + 5, lab, fontsize=12, color="gray")
    plt.xlabel("Number of rounds", fontsize=16)
    plt.ylabel(r"Complexity ($\log_2$ field operations)", fontsize=16)
    plt.title(rf"Vision, $s=2$ ($\omega={omega}$)", fontsize=18)
    plt.xlim(0, max_r)
    plt.ylim(0, ymax)
    plt.xticks(range(0, max_r + 1, 4))
    plt.grid(alpha=.3)
    plt.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.savefig(fn, dpi=200)
    plt.close()
    return rs, cur


# ---------------------------------------------------------------------------
# XHash12
# ---------------------------------------------------------------------------

def xhash_curves(max_steps=6, alpha=7, omega=2.81):
    out = {k: [] for k in ("s", "gb", "m1", "m3_k1", "m3_k2", "m3_k3", "m3_k4",
                           "m1_k2", "m1_k3", "m1_k4")}
    for s in range(1, max_steps + 1):
        n = 12 * s
        out["s"].append(s)
        out["gb"].append(xhash_gb(n, 1, alpha, omega))
        out["m1"].append(xhash_method1(n, 1, alpha, omega)[0])
        for k in (1, 2, 3, 4):
            out[f"m3_k{k}"].append(xhash_method3(n, k, alpha, omega, D0=alpha)["total"])
            if k > 1:
                out[f"m1_k{k}"].append(xhash_method1(n, k, alpha, omega)[0])
    return out


def plot_xhash(fn="xhash_complexity.png", max_steps=6, omega=2.81):
    plt.rcParams.update(SERIF)
    r = xhash_curves(max_steps, 7, omega)
    plt.figure(figsize=(11, 6.5))
    plt.plot(r["s"], r["gb"], "o-", color="orange", lw=2, label="CICO-1 (Groebner)")
    plt.plot(r["s"], r["m1"], "s-", color="blue", lw=2, label="CICO-1 (Method 1)")
    plt.plot(r["s"], r["m3_k1"], "^-", color="green", lw=2, label="CICO-1 (Method 3)")
    for k, col, mk in [(2, "red", "D"), (3, "purple", "p"), (4, "brown", "*")]:
        plt.plot(r["s"], r[f"m3_k{k}"], mk + "--", color=col, lw=2,
                 label=f"CICO-{k} (Method 1+3, P1/P2/P3)")
        plt.plot(r["s"], r[f"m1_k{k}"], mk + ":", color=col, lw=1.5, alpha=.6,
                 label=f"CICO-{k} (Method 1, direct)")
    plt.xlabel("Number of steps", fontsize=14)
    plt.ylabel(r"Complexity ($\log_2$ field operations)", fontsize=14)
    plt.title(rf"XHash12 ($\alpha=7$, $\omega={omega}$)", fontsize=16)
    plt.xticks(range(1, max_steps + 1))
    plt.grid(ls="--", alpha=.3)
    plt.legend(loc="upper left", fontsize=9, ncol=2)
    plt.tight_layout()
    plt.savefig(fn, dpi=200)
    plt.close()
    return r


# ---------------------------------------------------------------------------
# Poseidon
# ---------------------------------------------------------------------------

def dixon_cost(degs, n_vars, omega):
    m = len(degs)
    if m == 0:
        return 0.0
    M = dixon_support_size(degs)
    k = max(1, n_vars - (m - 1))
    return step4_hnf(M, k, max(1, sum(degs)), omega)


def poseidon_curves(t=8, c=3, d=3, alpha=3, RF=6, omega=2.81,
                    rp_range=range(2, 25, 2)):
    def io(rp):      return d, alpha ** (RF + rp)
    def sub_i(rp):
        k = 0 if rp < t - (c + d) else rp - (t - (c + d))
        return t, max(1, floor(sqrt(d / t) * alpha ** (RF / 2 + k / 2)))
    def sub_i2(rp):
        k = 0 if rp < t - (c + d) else rp - (t - (c + d))
        return d, max(1, floor(sqrt(d / t) * alpha ** (RF / 2 + k / 2)))
    def sub_o(rp):
        return 2 * t, max(1, int(alpha ** (RF / 2 + floor((rp - t + 1) / 2) / 2)))
    out = {k: [] for k in ("rp", "f4_io", "f4_sub_o", "f4_sub_i",
                           "dx_io", "dx_sub_o", "dx_sub_i")}
    for rp in rp_range:
        out["rp"].append(rp)
        n1, d1 = io(rp)
        out["f4_io"].append(gb_f4(n1, n1 * (d1 - 1) + 1, omega))
        out["dx_io"].append(dixon_cost([d1] * n1, n1, omega))
        n2, d2 = sub_o(rp)
        out["f4_sub_o"].append(gb_f4(n2, n2 * (d2 - 1) + 1, omega))
        out["dx_sub_o"].append(dixon_cost([d2] * n2, n2, omega))
        na, da = sub_i(rp)
        nb, db = sub_i2(rp)
        out["f4_sub_i"].append(gb_f4(na + nb,
                                     (na * (da - 1) + 1) + (nb * (db - 1) + 1) - 1,
                                     omega))
        out["dx_sub_i"].append(dixon_cost([da] * na + [db] * nb, na + nb, omega))
    return out


def plot_poseidon(fn="poseidon_complexity.png", omega=2.81, **kw):
    plt.rcParams.update(SERIF)
    r = poseidon_curves(omega=omega, **kw)
    plt.figure(figsize=(10, 6))
    for key, col, lab in [("f4_io", "blue", "F4 + I/O"),
                          ("f4_sub_o", "red", "F4 + subspace-O"),
                          ("f4_sub_i", "green", "F4 + subspace-I")]:
        plt.plot(r["rp"], r[key], "o-", color=col, lw=2, label=lab)
    for key, col, lab in [("dx_io", "blue", "Dixon + I/O"),
                          ("dx_sub_o", "red", "Dixon + subspace-O"),
                          ("dx_sub_i", "green", "Dixon + subspace-I")]:
        plt.plot(r["rp"], r[key], "s--", color=col, lw=2, label=lab)
    plt.xlabel(r"$R_P$ (number of partial rounds)", fontsize=15)
    plt.ylabel(r"Complexity ($\log_2$ field operations)", fontsize=15)
    plt.title(rf"Poseidon ($t=8$, $c=d=3$, $\alpha=3$, $\omega={omega}$)", fontsize=17)
    plt.legend(loc="upper left", fontsize=12)
    plt.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(fn, dpi=200)
    plt.close()
    return r


if __name__ == "__main__":
    for om in (2.0, 2.81):
        print("=" * 108)
        print(f"VISION  (omega = {om})   all model variants")
        print("=" * 108)
        rs, cur = vision_curves(15, om)
        hdr = f"{'r':>3} |" + "".join(f"{k:>13}" for k in cur) + "  | dominant"
        print(hdr)
        for i, r in enumerate(rs):
            st, _ = vision_stages(r, om, "full", "bezout")
            dom = max(st, key=lambda t: t[1])[0]
            print(f"{r:>3} |" + "".join(f"{cur[k][i]:>13.1f}" for k in cur)
                  + f"  | {dom}")
        print()

    print("=" * 108)
    print("XHASH12 (alpha=7, omega=2.81)")
    print("=" * 108)
    x = xhash_curves(6)
    print(f"{'s':>2} |{'GB':>9}{'M1(k=1)':>9}{'M3(k=1)':>9}"
          f"{'M3(k=2)':>10}{'M1(k=2)':>10}{'M3(k=3)':>10}{'M1(k=3)':>10}")
    for i, s in enumerate(x["s"]):
        print(f"{s:>2} |{x['gb'][i]:>9.1f}{x['m1'][i]:>9.1f}{x['m3_k1'][i]:>9.1f}"
              f"{x['m3_k2'][i]:>10.1f}{x['m1_k2'][i]:>10.1f}"
              f"{x['m3_k3'][i]:>10.1f}{x['m1_k3'][i]:>10.1f}")

    plot_vision("vision_complexity_all.png")
    plot_vision("vision_complexity_cameraready.png", series=CAMERA_READY)
    plot_vision("vision_complexity_all_w281.png", omega=2.81)
    plot_xhash()
    plot_poseidon()
    print("\nwrote vision_complexity_all.png, vision_complexity_cameraready.png,")
    print("      vision_complexity_all_w281.png, xhash_complexity.png, poseidon_complexity.png")


VISION  (omega = 2.00000000000000)   all model variants
  r |      ao_comp  bez_generic     bez_rank    bez_merge   mh_generic      mh_rank     mh_merge    gb_theory  | dominant
  2 |         25.0         28.2         24.1         24.1         27.9         18.2         18.2         41.1  | MERGE
  3 |         37.0         61.1         31.0         31.0         45.4         24.6         24.6         69.5  | MERGE
  4 |         49.0         77.1         48.1         48.1         61.1         36.7         36.2         98.0  | MERGE
  5 |         61.0        125.0         55.0         55.0         93.0         42.6         42.6        126.6  | MERGE
  6 |         73.0        141.0         72.1         72.1        109.0         54.6         54.2        155.2  | MERGE
  7 |         85.0        189.0         79.0         79.0        141.0         60.6         60.6        183.9  | MERGE
  8 |         97.0        205.0         96.1         96.1        157.0         72.6         72.2        212.

In [4]:
"""make_vision_fig.py -- regenerate vision_complexity.png (Figure 4).

The Dixon curve is now the maximum over all stages of the meet-in-the-middle
chain (every forward/backward rung and the final merge), with the per-rung
cost split into Step 1 / Step 3 / Step 4 exactly as in Tables 10-11 of the
appendix.  The old curve hard-coded d_max = 2^(4r) for every r, which is
inconsistent with Table 9 for odd r.
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "STIXGeneral",
                     "font.serif": ["STIXGeneral", "DejaVu Serif"],
                     "mathtext.fontset": "stix"})

MAXR = 27
rs = list(range(2, MAXR))
dixon = [max(c for _, c in vision_stages(r, 2.0, "full", "bezout")[0]) for r in rs]
gbt = [vision_gb_theory(r, 2.0) for r in rs]
ks = sorted(GB_OVERDEF)

plt.figure(figsize=(9, 5.6))
plt.plot(rs, dixon, "o-", color="red", lw=2, ms=4, label="Dixon resultant")
plt.plot(rs, gbt, "s-", color="royalblue", lw=2, ms=4, alpha=.85,
         label="Gr\u00f6bner (theoretical)")
plt.plot(ks, [GB_OVERDEF[k] for k in ks], "^-", color="green", lw=2, ms=5,
         label="Gr\u00f6bner (Overdefined)")
for y, lab in [(128, "128-bit"), (256, "256-bit")]:
    plt.axhline(y, color="gray", ls="--", lw=1, alpha=.7)
    plt.text(MAXR - 4.5, y + 4, lab, fontsize=13, color="gray")
plt.xlabel("Number of Rounds", fontsize=18)
plt.ylabel(r"Complexity ($\log_2$ field operations)", fontsize=18)
plt.title(r"Complexity comparison for Vision ($\omega = 2$)", fontsize=20)
plt.xlim(0, MAXR - 1)
plt.ylim(0, 350)
plt.xticks(range(0, MAXR, 4), fontsize=14)
plt.yticks(fontsize=14)
plt.grid(alpha=.3)
plt.legend(loc="lower right", fontsize=13)
plt.tight_layout()

if __name__ == "__main__":
    # In Jupyter, sys.argv contains the kernel's `-f` connection argument;
    # use the notebook's current directory rather than treating it as a path.
    plt.savefig("vision_complexity.png", dpi=200)
    print([(r, round(c, 1)) for r, c in zip(rs, dixon)][:8])


[(2, 24.1), (3, 31.0), (4, 48.1), (5, 55.0), (6, 72.1), (7, 79.0), (8, 96.1), (9, 103.0)]


In [5]:
"""make_xhash_fig.py -- regenerate xhash_complexity.png (Figure 5).

Two corrections with respect to AO_Complexity.ipynb:

 (1) the hybrid curves for k >= 2 now cost all three phases (P1 reduction,
     P2 successive elimination, P3 final Dixon) instead of P3 alone;
 (2) the direct Dixon method is plotted for every k, not only for k = 1, so
     that the crossover between the two strategies is visible.
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "STIXGeneral",
                     "font.serif": ["STIXGeneral", "DejaVu Serif"],
                     "mathtext.fontset": "stix"})

ALPHA, OMEGA, MAXS = 7, 2.81, 6
steps = list(range(1, MAXS + 1))

gb = [xhash_gb(12 * s, 1, ALPHA, OMEGA) for s in steps]
m1 = {k: [xhash_method1(12 * s, k, ALPHA, OMEGA)[0] for s in steps]
      for k in (1, 2, 3, 4)}
m3 = {k: [xhash_method3(12 * s, k, ALPHA, OMEGA, D0=ALPHA)["total"] for s in steps]
      for k in (1, 2, 3, 4)}

plt.figure(figsize=(10, 6))
plt.plot(steps, gb, "o-", color="orange", lw=2, label="CICO-1 (Gr\u00f6bner basis)")
plt.plot(steps, m1[1], "s-", color="blue", lw=2, label="CICO-1 (Method 1)")
plt.plot(steps, m3[1], "^-", color="green", lw=2, label="CICO-1 (Method 3)")
for k, col, mk in [(2, "red", "D"), (3, "purple", "p"), (4, "brown", "*")]:
    plt.plot(steps, m3[k], mk + "--", color=col, lw=2,
             label=f"CICO-{k} (Method 1+3)")
    plt.plot(steps, m1[k], mk + ":", color=col, lw=1.6, alpha=.65,
             label=f"CICO-{k} (Method 1)")

plt.xlabel("Number of Steps", fontsize=18)
plt.ylabel(r"Complexity ($\log_2$ field operations)", fontsize=18)
plt.title(r"Complexity comparison for XHash12 ($\alpha=7$, $\omega = 2.81$)",
          fontsize=20)
plt.xticks(steps)
plt.tick_params(axis="both", labelsize=14)
plt.grid(ls="--", alpha=.35)
plt.legend(loc="upper left", fontsize=13, ncol=2)
plt.tight_layout()

if __name__ == "__main__":
    # Ignore Jupyter's kernel arguments and save beside the notebook.
    plt.savefig("xhash_complexity.png", dpi=200)
    print("ok")


ok
